In [1]:
import cProfile

from poisson_hypergraph import GH
from NMI_func import NMI
import xgi
import numpy as np
import networkx as nx
import math
from numba import jit, njit

# function that generates an intstance of the class GH with the parameters stored in true_theta and for timesteps times resulting in a graph with timesteps+1 edges
# the graph starts with 2 nodes but will have more as novel nodes are added
def generate_graph(true_theta, timesteps):
    true_p, true_q, gamma_nu, gamma_nr, gamma_eu, gamma_er = true_theta
    H = xgi.Hypergraph([[0, 1]])
    H.set_node_attributes({0 : 0, 1 : 1}, name = "label")
    g = GH(H, [0, 1], true_p, true_q)
    g.add_hyperedge(timesteps, gamma_nu, gamma_nr, gamma_eu, gamma_er)
    return g

def generate_graph_26_starting_nodes(true_theta, timesteps):
    true_p, true_q, gamma_nu, gamma_nr, gamma_eu, gamma_er = true_theta
    H = xgi.Hypergraph([[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25]])
    H.set_node_attributes({0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:1,14:1,15:1,16:1,17:1,18:1,19:1,20:1,21:1,22:1,23:1,24:1,25:1}, name="label")
    g=GH(H, [0,1], true_p, true_q)
    g.add_hyperedge(timesteps, gamma_nu, gamma_nr, gamma_eu, gamma_er)

    return g

true_p = .9
true_q = .1
# gamma_nu is poisson weight for the distribution for number of like labeled novel nodes of choosen u novel nodes added
# gamma_nr is poisson weight for the distribution for number of opposite labeled nodes of choosen u novel nodes added
# gamma_eu is poisson weight for the distribution for number of like labeled nodes of choosen u external nodes added
# gamma_er is poisson weight for the distribution for number of opposite labeled nodes of choosen u external nodes added
gamma_nu, gamma_nr, gamma_eu, gamma_er = .01, .01, 1, 0.25

# should be in this order
true_theta = [true_p, true_q, gamma_nu, gamma_nr, gamma_eu, gamma_er]
timesteps = 10

g = generate_graph(true_theta, timesteps)

# input is an instance of the class GH defined in poisson_hypergraph.py... output a tuple of lists of total likelihood and nmi indexed by timestep
def greedy_community_detection_algo(g, generate_likelihoods):
    total_likelihoods = []
    nmis = []

    null_labels = np.random.choice([0,1], size=len(g.get_labels()))
    true_labels = g.get_labels()

    greedy_steps = 1000
    step_num = 0
    while step_num < greedy_steps:
        delta_E = 0
        e_index = np.random.choice(range(1, len(g.get_edges())))
        
        v_index = np.random.choice(list(g.get_edges()[e_index]))

        canidate_f = []
        for f_index in range(e_index):
            if len(g.get_edges()[f_index].intersection(g.get_edges()[e_index])) != 0:
                canidate_f.append(f_index)


        for f_index in canidate_f:
            delta_E += g.greedy_expectation_step_given_f(v_index, f_index, e_index, true_theta, null_labels) / len(canidate_f)
        
        if (delta_E > 0):
            null_labels[v_index] = 1 - null_labels[v_index]
            # print("swapped label of " + str(v_index) + " from edge " + str(e_index))

        step_num+=1

        # if (step_num % 25 == 0):
        if generate_likelihoods:
            total_likelihoods.append(g.total_log_likelihood(true_theta, null_labels))
            nmis.append(NMI(g.get_labels(), null_labels, g))
        
    # print("greedy labels likelihood: " + str(g.expected_log_likelihood_total(true_theta, null_labels)))
    # print("true labels likelihood: " + str(g.expected_log_likelihood_total(true_theta,true_labels)))
    # print(null_labels)
    # print(true_labels)
    if generate_likelihoods:
        return total_likelihoods, nmis
    else:
        return null_labels

def greedy_community_detection_algo_with_posterior_prob(g, generate_likelihoods):
    null_labels = np.random.choice([0,1], size=len(g.get_labels()))
    true_labels = g.get_labels()

    greedy_steps = 2000
    step_num = 0

    # TODO remove, for testing
    total_likelihoods = []
    nmis = []
    while step_num < greedy_steps:
        delta_E = 0
        e_index = np.random.choice(range(0, len(g.get_edges())))
        v_index = np.random.choice(list(g.get_edges()[e_index]))

        new_labels = null_labels.copy()
        new_labels[v_index] = 1 - new_labels[v_index]

        f_probs = g.f_prob_array_given_e(e_index, true_theta, new_labels)

        canidate_f = []
        for f_index in range(e_index):
            if len(g.get_edges()[f_index].intersection(g.get_edges()[e_index])) != 0:
                canidate_f.append(f_index)

        for f_index in canidate_f:
            delta_E += g.greedy_expectation_step_given_f(v_index, f_index, e_index, true_theta, null_labels) * f_probs[f_index]


        
        if (delta_E > 0):
            null_labels[v_index] = 1 - null_labels[v_index]
        
        # if (step_num % 25 == 0):
        if generate_likelihoods:
            total_likelihoods.append(g.total_log_likelihood(true_theta, null_labels))
            nmis.append(NMI(g.get_labels(), null_labels, g))

            
        step_num+=1

    if generate_likelihoods:
        return total_likelihoods, nmis
    else:
        return null_labels

def greedy_community_detection_algo_3_label(g, generate_likelihoods):
    NUM_ES = 3

    null_labels = np.random.choice([0,1], size=len(g.get_labels()))
    true_labels = g.get_labels()

    greedy_steps = 2000
    step_num = 0

    # TODO remove, for testing
    total_likelihoods = []
    nmis = []
    while step_num < greedy_steps:
        delta_E = 0
        
        v_index = np.random.choice(list(range(len(g.get_labels()))))

        canidate_e = []
        for e_index in range(1, len(g.get_edges())):
            if v_index in g.get_edges()[e_index]:
                canidate_e.append(e_index)

        e_indexes = np.random.choice(canidate_e, min(NUM_ES, len(canidate_e)))

        new_labels = null_labels.copy()
        new_labels[v_index] = 1 - new_labels[v_index]

        for e_index in e_indexes:
            f_probs = g.f_prob_array_given_e(e_index, true_theta, new_labels)

            canidate_f = []
            for f_index in range(e_index):
                if len(g.get_edges()[f_index].intersection(g.get_edges()[e_index])) != 0:
                    canidate_f.append(f_index)

            for f_index in canidate_f:
                delta_E += g.greedy_expectation_step_given_f(v_index, f_index, e_index, true_theta, null_labels) * f_probs[f_index]


        if (delta_E > 0):
            null_labels[v_index] = 1 - null_labels[v_index]
        
        # if (step_num % 25 == 0):
        if generate_likelihoods:
            total_likelihoods.append(g.total_log_likelihood(true_theta, null_labels))
            nmis.append(NMI(g.get_labels(), null_labels, g))

            
        step_num+=1

    if generate_likelihoods:
        return total_likelihoods, nmis
    else:
        return null_labels

In [ ]:
timesteps = 100
def test():
    for _ in range(1):
        g = generate_graph(true_theta, timesteps)
        # False specifies whether to generate true log likelihood every step for graphing

        # both lines below call the different versions of the algo... 
        # 3_label has been the best in testing but takes longer



        # greedy_community_detection_algo_with_posterior_prob(g, False)
        greedy_community_detection_algo_3_label(g, False)


cProfile.run('test()', sort='ncalls')

# Some Thoughts from Phil

So in the profile call above, it looks like we spend most of our time forming the distribution over the preceding edges $f$ and then evaluating the greedy update. I wonder whether we might be able to do better here (in terms of performance) with array programming. Here's the kind of thing I mean. First, we'll define a function that grabs only the edges of $H$ that have nonempty intersections with a given edge $e$. 

In [3]:
def edge_neighborhood(H, e_ix, prior_only = False,as_node_sets = False):
    """
    """
    
    e = H.edges.members(e_ix)
    neighbor_edges = []
    
    for i in e: 
        for e_ix_ in H.nodes.memberships(i):
            if ((not prior_only) or (e_ix_ < e_ix)) and (e_ix_ != e_ix): 
                neighbor_edges.append(e_ix_)

    neighbor_edges = set(neighbor_edges)
    if as_node_sets: 
        return [H.edges.members(e_ix_) for e_ix_ in neighbor_edges]
    else: 
        return neighbor_edges

For illustrative purposes, let's generate a hypergraph: 

In [4]:
theta = [0.8, 0.2, 1.0, 0.5, 0.1, 0.1]
g = generate_graph(theta, timesteps = 1000)

Now we'll pick an edge and consider the edges that intersect with it. 

In [14]:

# edge that I am going to randomly select
e_ix = 50 
e = g.H.edges.members(e_ix)

print(f"edge {e_ix}: {e}")

candidate_fs = edge_neighborhood(g.H, e_ix, prior_only = True, as_node_sets = True)

print(f"candidate f's for edge {e_ix} (nonempty intersection):")

for f in candidate_fs: 
    print(f)

edge 50: {1, 55, 56, 57, 58}
candidate f's for edge 50 (nonempty intersection):
{0, 1}
{1}
{1}
{0, 1, 2}
{0, 1, 3}
{1, 3, 7}
{1, 14}
{1, 15}
{1}
{1, 18}
{3, 1, 19, 7}
{1, 3, 21, 22}
{1, 26}
{1, 3, 21, 22}
{1, 29, 30}
{1, 34}
{1, 37, 38, 39, 40, 41}
{1, 44, 15}
{1, 21, 3, 45}
{1, 3, 22}
{1, 30}


In [11]:
def slow_accumulator(theta, candidate_fs, e):
    accumulator = 0
    for f in candidate_fs:
        intersection = len(f.intersection(e))
        difference = len(f.symmetric_difference(e))
        accumulator += theta[0]**intersection * (1-theta[0])**difference
    return accumulator

@jit(nopython=True)
def jit_accumulator(theta, candidate_fs, e):
    accumulator = 0
    for f in candidate_fs:
        intersection = len(f.intersection(e))
        difference = len(f.symmetric_difference(e))
        accumulator += theta[0]**intersection * (1-theta[0])**difference
    return accumulator

def numpy_accumulator(theta, candidate_fs, e):
    intersections = np.array([len(f.intersection(e)) for f in candidate_fs])
    differences = np.array([len(f.symmetric_difference(e)) for f in candidate_fs])
    return np.sum(theta[0]**intersections * (1-theta[0])**differences)

In [13]:
from numba.typed import List

nb_list = List()
for f in candidate_fs:
    nb_list.append(f)

%timeit slow_accumulator(theta, candidate_fs, e)
%timeit jit_accumulator(theta, nb_list, e)
%timeit numpy_accumulator(theta, candidate_fs, e)

12.3 μs ± 69 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
15.3 μs ± 39.2 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
19.6 μs ± 78.1 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [ ]:
%%timeit

random_accumulator = 0

for f in candidate_fs: 
    random_accumulator += theta[0]**(len(f.intersection(e)))*(1-theta[0])**(len(f.symmetric_difference(e)))

637 ns ± 2.19 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [ ]:
%%timeit 

intersections = np.array([len(f.intersection(e)) for f in candidate_fs])
differences = np.array([len(f.symmetric_difference(e)) for f in candidate_fs])
random_accumulator_2 = theta[0]**intersections * (1-theta[0])**differences

6.52 μs ± 49.3 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [ ]:
g.H.edges.members()

[{0, 1},
 {1, 2},
 {0},
 {1, 3, 4},
 {2, 5, 6},
 {0, 1, 5},
 {0, 5, 7, 8},
 {0, 9, 10},
 {0, 5, 7, 11, 12},
 {12},
 {2, 5, 13, 14},
 {0},
 {4, 15, 16, 17},
 {2, 5, 13, 18, 19},
 {0, 20, 21},
 {0, 22, 23},
 {4, 15, 16, 24, 25},
 {15, 17, 26, 27},
 {6, 28},
 {8, 29},
 {4, 16},
 {8, 29, 30},
 {17, 26, 27, 31, 32},
 {17, 26, 27, 31, 33, 34},
 {0, 35},
 {4, 7, 15, 36, 37},
 {4, 6, 38, 39, 40, 41},
 {13, 18, 19, 40, 42},
 {8},
 {0, 43, 44},
 {17, 27, 31, 33, 34},
 {4, 15, 16},
 {2, 5, 35, 36, 45, 46},
 {27, 34, 47},
 {17, 26, 27, 31},
 {18, 19, 40, 48, 49, 50},
 {2, 5, 35, 36, 51, 52},
 {27, 31, 33, 34, 53, 54, 55, 56},
 {0, 5, 6, 7, 8, 57, 58, 59},
 {0, 60, 61},
 {6, 38, 39, 41, 62},
 {5, 35, 36, 52, 63},
 {35, 36, 51, 64, 65},
 {6, 38, 40, 41, 66, 67, 68},
 {2, 5, 69, 70, 71, 72},
 {8, 29, 30, 73},
 {0, 21, 74, 75, 76},
 {2, 5, 69, 77, 78},
 {41, 67, 68, 79, 80},
 {0, 5, 6, 7, 57},
 {8, 29},
 {0, 5, 8, 81},
 {6, 38, 40, 41, 58},
 {43, 44},
 {8, 29},
 {2, 5, 35, 36, 51, 82, 83},
 {4, 15, 16